# GDELT — Rare Earth Sentiment Data
**Master's Thesis | BSE Financial Economics**  
**Authors:** Aleix Cuevas, Niklas Biemel, Nicolas Koutalakis  

---
This notebook extracts, cleans and explores news sentiment data from the **GDELT Global Knowledge Graph (GKG)** via Google BigQuery.  
The goal is to build two sentiment series for Nd, Pr, Tb and Dy (rare earth metals):  
- **Tone**: weighted average tone of articles (−100 to +100. Practically oscillates between −10 and +10)
- **Article Growth**: growth rate of article volume (proxy for media attention / Buzz)  

Methodology follows **Guidolin & Pedio (2020)** — *Media Attention vs. Sentiment as Drivers of Conditional Volatility Predictions: An Application to Brexit*.

---
## Architecture: cache-once, filter locally

To minimise BigQuery costs while allowing free experimentation with different filters, this notebook follows a **two-stage architecture**:

1. **One paid BigQuery scan** with a deliberately permissive filter (`keyword_filter_broad` with `n_keywords=1`) extracts an article-level dataset where every potentially relevant article is included. For each article we pre-compute a set of boolean columns (one per keyword concept), so that any subsequent filter can be reconstructed offline.
2. **All subsequent filtering, comparison, inspection and aggregation** happens locally in pandas at zero cost. Iterating on the filter logic, comparing thresholds, sampling articles for manual inspection and producing the final daily series — all of this runs against the cached dataframe.

# Sample Period Justification

## Sample Period: April 1st, 2015 – April 1st, 2026

### Start date: April 1st, 2015
GDELT 2.0 was launched in February 2015. We extract data from April 1st onwards (skipping only February and March, the launch ramp-up months) so that the cache contains the full universe of GDELT 2.0 observations for our topic. Whether 2015 is finally included in the econometric analysis or treated as a warm-up period that is dropped *ex post* is a decision we leave to the local-filtering stage, once we can inspect the actual daily article counts in 2015 and compare them to subsequent years.

### End date: April 1st, 2026
The sample extends to the most recent month with complete GDELT data available at the time of extraction (April 2026). This window deliberately covers a sequence of events that are highly relevant for rare earth markets:

- **2016–2017**: Post-collapse stabilisation of REE prices following the 2011–2015 speculative cycle.
- **2018–2019**: US–China trade war and explicit rare earth export threats by Beijing as retaliation against US tariffs.
- **2020**: COVID-19 pandemic and global supply chain disruptions.
- **2021–2022**: Post-pandemic demand recovery and the Russia–Ukraine war, which intensified the geopolitical premium on critical minerals.
- **2023–2024**: REE price decline driven by weak Chinese domestic demand and oversupply.
- **2025**: China's rare earth export controls (April 2025), the most significant supply restriction since the 2010 embargo against Japan.
- **Early 2026**: Continued geopolitical tensions and renewed US–China trade negotiations.

### Total sample
The extraction window spans **11 calendar years** (~4,019 days). Daily series are aggregated to weekly frequency where required to match the availability of physical REE price data from the SMM database.

## 0. Setup — Install & Authenticate
Make sure your Google Cloud project has **BigQuery API enabled**.

In [2]:
# Install the Google Cloud client libraries into THIS Jupyter kernel.
# %pip is preferred over !pip — it targets the same Python interpreter
# that is running the notebook.
%pip install --quiet google-cloud-bigquery pandas-gbq pyarrow db-dtypes pydata-google-auth

# ──────────────────────────────────────────────────────────────────────
# Authentication: browser-based OAuth via pydata_google_auth.
#
# Why this and not `google.auth.default()`?
#   - google.auth.default() relies on Application Default Credentials
#     that are normally provisioned by the Google Cloud SDK (gcloud),
#     which is NOT installed on this machine.
#   - pydata_google_auth does the OAuth flow itself: it opens a browser
#     the first time you run it, you sign in with your Google account,
#     and the credentials are cached at
#         ~/.config/pydata/pydata_google_credentials.json
#     so future kernel sessions reuse them silently.
# ──────────────────────────────────────────────────────────────────────
import pydata_google_auth

SCOPES = ['https://www.googleapis.com/auth/bigquery']
credentials = pydata_google_auth.get_user_credentials(
    SCOPES,
    auth_local_webserver=True,   # spins up a temporary local webserver for the OAuth redirect
)
print('Authentication OK. Credentials cached for future runs.')

Note: you may need to restart the kernel to use updated packages.
Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=262006177488-3425ks60hkk80fssi9vpohv88g6q1iqd.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbigquery&state=vqMfhsfvvvCedDmDTL937jl2Y2H9L8&code_challenge=s1_jNB1E9GmlcTqvfaXkYqDkR_F4bM56PtcuHYgBXd4&code_challenge_method=S256&prompt=consent&access_type=offline


Unable to create credentials directory.


Authentication OK. Credentials cached for future runs.


In [3]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from google.cloud import bigquery

# -------------------------------------------------------
# CONFIGURATION — edit only this block
# -------------------------------------------------------
PROJECT_ID = 'gdelt-thesis-487419'
START_DATE = '20150401'
END_DATE   = '20260401'
# -------------------------------------------------------

# Pass the OAuth credentials from the previous cell explicitly — the
# default `bigquery.Client(project=PROJECT_ID)` would try ADC again and
# fail with DefaultCredentialsError.
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print(f'BigQuery client ready. Project: {PROJECT_ID}')
print(f'Sample period: {START_DATE} → {END_DATE}')

/Applications/anaconda3/lib/python3.8/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.8.8). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


BigQuery client ready. Project: gdelt-thesis-487419
Sample period: 20150401 → 20260401


/Applications/anaconda3/lib/python3.8/site-packages/google/cloud/bigquery/__init__.py:129: FutureWarning: The python-bigquery library no longer supports Python 3.7 and Python 3.8. Your Python version is 3.8.8. We recommend that you update soon to ensure ongoing support. For more details, see: [Google Cloud Client Libraries Supported Python Versions policy](https://cloud.google.com/python/docs/supported-python-versions)
  warnings.warn(


# 1. Shared utilities
Common helpers used by every filter: regex builder, the GDELT search blob, and a small toolkit for building filter SQL strings.

In [4]:
# Regex helper — turns a list of plain-language terms into a single alternation
# pattern. Spaces inside terms become flexible whitespace/hyphen/underscore so
# 'rare earth' matches 'rare earth', 'rare-earth', 'rare_earth', etc.
def regex_union(terms):
    return r'(?:' + '|'.join(
        re.escape(t.lower()).replace(r'\ ', r'[\s\-_]+') for t in terms
    ) + r')'

# GDELT search blob: lowercase concatenation of the curated text fields.
# Should we eclude DocumentIdentifier (URL) and V2Persons?
#   - URLs cause e-commerce / product-page false positives (viewngr.com etc.)
#   - V2Persons is largely subsumed by AllNames and adds little signal.
SEARCH_BLOB = ("LOWER(CONCAT("
    "IFNULL(V2Themes,''),' | ',IFNULL(V2Organizations,''),' | ',"
    "IFNULL(V2Persons,''),' | ',IFNULL(AllNames,''),' | ',"
    "IFNULL(DocumentIdentifier,'')))")

# 2. Filters
We define two filters with different logic. **Both are evaluated on the same article-level cache** that we pull below — so we can compare them, switch between them, or invent new ones, all without paying BigQuery again.

In [5]:
# -----------------------------------------------------------
# FILTER — Concept-grouped co-occurrence counter
# Each concept contributes at most 1 to the score, regardless of
# how many of its synonyms appear in the article. The article passes
# if the score is >= n_keywords.
# -----------------------------------------------------------
KEYWORD_CONCEPTS = {
    # ---- Magnet REE metals (the four target metals of the original analysis) ----
    'neodymium':        ['neodymium'],
    'praseodymium':     ['praseodymium'],
    'dysprosium':       ['dysprosium'],
    'terbium':          ['terbium'],
    'ndpr':             ['ndpr'],
    'ndfeb':            ['ndfeb'],

    # ---- Other commercially relevant REEs (light + heavy) ----
    # Optional: include if you want broader REMX-sector coverage.
    # Treat as separate concepts so they can be filtered out locally if needed.
    'lanthanum':        ['lanthanum'],
    'cerium':           ['cerium'],
    'samarium':         ['samarium'],
    'europium':         ['europium'],
    'gadolinium':       ['gadolinium'],
    'yttrium':          ['yttrium'],

    # ---- Rare earth phrase (all spelling/plural/element/magnet variants) ----
    'rare_earth':       ['rare earth', 'rare earths', 'rare-earth', 'rare-earths',
                         'rare earth element', 'rare earth elements',
                         'rare-earth element', 'rare-earth elements',
                         'rare earth magnet', 'rare earth magnets',
                         'rare-earth magnet', 'rare-earth magnets',
                         'rare earth metal', 'rare earth metals',
                         'rare-earth metal', 'rare-earth metals'],

    # ---- REMX constituent companies (unambiguous names only) ----
    # These capture news that moves the ETF without explicitly mentioning REE.
    'lynas':            ['lynas', 'lynas rare earths', 'lynas corporation'],
    'mp_materials':     ['mp materials'],
    'jl_mag':           ['jl mag', 'jl mag rare-earth'],
    'arafura':          ['arafura resources', 'arafura rare earths'],
    'china_northern':   ['china northern rare earth'],

    # ---- ETF ticker itself (low signal, kept for completeness) ----
    'remx':             ['remx'],
}

def build_filter(concepts, n_keywords, search_blob):
    """Build the SQL for the filter with a given threshold."""
    return "(" + " + ".join(
        f"CAST(REGEXP_CONTAINS({search_blob}, r'\\b{regex_union(group)}\\b') AS INT64)"
        for group in concepts.values()
    ) + f") >= {n_keywords}"

# Used as a BROAD pre-filter for the cache: any single concept matches.
# This is the most permissive setting — we'll narrow it locally afterwards.
keyword_filter_broad = build_filter(KEYWORD_CONCEPTS, n_keywords=1,
                                        search_blob=SEARCH_BLOB)

print(f'Total concepts: {len(KEYWORD_CONCEPTS)}')
print(f'Concept names:  {list(KEYWORD_CONCEPTS.keys())}')

Total concepts: 19
Concept names:  ['neodymium', 'praseodymium', 'dysprosium', 'terbium', 'ndpr', 'ndfeb', 'lanthanum', 'cerium', 'samarium', 'europium', 'gadolinium', 'yttrium', 'rare_earth', 'lynas', 'mp_materials', 'jl_mag', 'arafura', 'china_northern', 'remx']


# 3. Cache query — extract article-level data
We run **a single broad query** that fetches one row per article over the full sample period. For each article we store:

- `date_str` — date of the article
- `url` — DocumentIdentifier (for manual inspection)
- `tone`, `pos_score`, `neg_score`, `polarity` — V2Tone components
- `has_<concept>` — one boolean per keyword concept (7 columns)

The pre-filter is `keyword_filter_broad` (`n_keywords=1`), so the cache is a strict superset of anything any other variant filter can produce. We can recover any of those filters offline by applying boolean logic on the cache.

In [6]:
# Build CACHE_QUERY: article-level rows + boolean columns for every concept.
concept_selects = [
    f"REGEXP_CONTAINS({SEARCH_BLOB}, r'\\b{regex_union(group)}\\b') AS has_{name}"
    for name, group in KEYWORD_CONCEPTS.items()
]
concept_selects_sql = ",\n    ".join(concept_selects)

CACHE_QUERY = f"""
SELECT
    SUBSTR(CAST(DATE AS STRING), 1, 8)                        AS date_str,
    DocumentIdentifier                                        AS url,
    SAFE_CAST(SPLIT(V2Tone, ',')[OFFSET(0)] AS FLOAT64)       AS tone,
    SAFE_CAST(SPLIT(V2Tone, ',')[OFFSET(1)] AS FLOAT64)       AS pos_score,
    SAFE_CAST(SPLIT(V2Tone, ',')[OFFSET(2)] AS FLOAT64)       AS neg_score,
    SAFE_CAST(SPLIT(V2Tone, ',')[OFFSET(3)] AS FLOAT64)       AS polarity,
    {concept_selects_sql}
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE
    _PARTITIONTIME >= TIMESTAMP('{START_DATE[:4]}-{START_DATE[4:6]}-{START_DATE[6:]}')
    AND _PARTITIONTIME <= TIMESTAMP('{END_DATE[:4]}-{END_DATE[4:6]}-{END_DATE[6:]}')
    AND DATE BETWEEN {START_DATE}000000 AND {END_DATE}235959
    AND {keyword_filter_broad}
"""

print('CACHE_QUERY ready.')

CACHE_QUERY ready.


In [7]:
# Dry-run to estimate cost (FREE — does not execute the query, only validates it)
MAX_BYTES_BILLED = int(3.0 * 1024**4)   # 3 TiB cap, comfortable margin
PRICE_USD_PER_TIB = 6.25

dry_run = client.query(
    CACHE_QUERY,
    job_config=bigquery.QueryJobConfig(
        dry_run=True,
        use_query_cache=False,
        maximum_bytes_billed=MAX_BYTES_BILLED,
    )
)

bytes_scanned = dry_run.total_bytes_processed
gb_scanned    = bytes_scanned / (1024**3)
tib_scanned   = bytes_scanned / (1024**4)
cost_upper    = tib_scanned * PRICE_USD_PER_TIB

print(f'Estimated scan: {gb_scanned:,.1f} GB ({tib_scanned:.4f} TiB)')
print(f'Upper-bound cost (before free tier): ${cost_upper:,.2f} USD')
print(f'Hard cap on real run: {MAX_BYTES_BILLED / (1024**4):.2f} TiB')

if tib_scanned <= 1.0:
    print('\n  Within the 1 TiB/month free tier (if not already consumed)')
else:
    print(f'\n  Exceeds free tier by {tib_scanned - 1.0:.2f} TiB')

Estimated scan: 2,578.8 GB (2.5184 TiB)
Upper-bound cost (before free tier): $15.74 USD
Hard cap on real run: 3.00 TiB

  Exceeds free tier by 1.52 TiB


In [12]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  REAL MONEY — this cell pays for the BigQuery scan.                  ║
# ║  Run it ONCE for the full sample period. Everything downstream is    ║
# ║  local pandas and costs nothing.                                     ║
# ╚══════════════════════════════════════════════════════════════════════╝

# real_config = bigquery.QueryJobConfig(
#     use_query_cache=False,
#     maximum_bytes_billed=MAX_BYTES_BILLED,
# )

# print('Executing CACHE_QUERY... (this may take 1–3 minutes)')
# df_cache = client.query(CACHE_QUERY, job_config=real_config).to_dataframe()

# print(f'\nDone. {len(df_cache):,} articles retrieved.')
# print(f'Date range: {df_cache["date_str"].min()} → {df_cache["date_str"].max()}')
# print(f'Memory used: {df_cache.memory_usage(deep=True).sum() / (1024**2):.1f} MB')
# df_cache.head()


# Save the cached article-level dataframe to the local sentiment-data folder.
# from pathlib import Path

# OUT_PATH = Path('Sentiment data') / 'SentimentData_Aleix_REMX.csv'
# OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# df_cache.to_csv(OUT_PATH, index=False)

# size_mb = OUT_PATH.stat().st_size / (1024 ** 2)
# print(f'CSV saved -> {OUT_PATH}  ({size_mb:.1f} MB, {len(df_cache):,} rows)')